In [91]:
! /usr/local/bin/python3.12 -m pip install tensorflow keras


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [92]:
import numpy
import pandas

from matplotlib import pyplot
import seaborn

# Redes neuronales
import keras

In [93]:
titanic3 = pandas.read_csv("../conjuntos/titanic3.csv")

titanic3

,Sex,Age,Pclass2,Family,Family2,Fare2,Cabin2,Embarked2,Survived2
0,male,22.0,3ra,1,couple,2.110213,no,S,no
1,female,38.0,1ra,1,couple,4.280593,yes,C,yes
2,female,26.0,3ra,0,single,2.188856,no,S,yes
3,female,35.0,1ra,1,couple,3.990834,yes,S,yes
4,male,35.0,3ra,0,single,2.202765,no,S,no
...,...,...,...,...,...,...,...,...,...
886,male,27.0,2da,0,single,2.639057,no,S,no
887,female,19.0,1ra,0,single,3.433987,yes,S,yes
888,female,NaN,3ra,3,family,3.196630,no,S,no
889,male,26.0,1ra,0,single,3.433987,yes,C,yes


In [95]:
pandas.__version__

'2.2.2'

In [101]:
pandas.get_dummies(titanic3[["Pclass2"]])[["Pclass2_1ra", "Pclass2_2da"]].astype(int)

,Pclass2_1ra,Pclass2_2da
0,0,0
1,1,0
2,0,0
3,1,0
4,0,0
...,...,...
886,0,1
887,1,0
888,0,0
889,1,0


In [ ]:
x1 = (titanic3["Sex"] == "female").astype(int)
x1.name = "Sex_female"
x2 = titanic3["Age"]
x3 = (titanic3["Pclass2"] == "1ra").astype(int)
x3.name = "Pclass2_1ra"
x4 = (titanic3["Pclass2"] == "2da").astype(int)
x3.name = "Pclass2_2da"
x5 = titanic3["Family"]
x6 = (titanic3["Family2"] == "couple").astype(int)
x6.name = "Family2_couple"
x7 = (titanic3["Family2"] == "family").astype(int)
x7.name = "Family2_family"
x8 = titanic3["Fare2"]
x9 = (titanic3["Cabin2"] == "yes").astype(int)
x10 = (titanic3["Embarked2"] == "C").astype(int)
x10.name = "Embarked2_C"
x11 = (titanic3["Embarked2"] == "Q").astype(int)
x11.name = "Embarked2_Q"
x12 = (titanic3["Embarked2"] == "X").astype(int)
x12.name = "Embarked2_X"

y1 = (titanic3["Survived2"] == "yes").astype(int)

indices_edades_nulas = x2[x2.isna()].index # Índices de las edades conocidas
indices_edades_no_nulas = x2.dropna().index # Índices de las edades no conocidas
Cy = x2.dropna() # Edades conocidas
Cx = pandas.DataFrame([x1, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, y1]).T.loc[indices_edades_no_nulas] # Clúster de edades conocidas
Cp = pandas.DataFrame([x1, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12, y1]).T.loc[indices_edades_nulas] # Clúster de predicción (para las edades no conocidas)

from sklearn.cluster import KMeans # MeanShift, AffinityPropagation

clu = KMeans(n_clusters=20) # Reconocer 20 tipos de clúster
clu.fit(Cx, Cy) # Entrenar el clúster
clusters = clu.predict(Cx) # Predecir el clúster para las edades conocidas
age_cluster = pandas.DataFrame({
    "Age": Cy.values, # Edades conocidas
    "Clu": clusters, # Clúster de las edades conocidas
}).groupby(["Clu"]).median() # Agrupadas y se extrae la mediana (para las edades conocidas)

X = pandas.DataFrame([x1, x2, x3, x4, x5, x6, x7, x8, x9, x10, x11, x12]).T # Las predictivas (sin imputar)
Y = pandas.DataFrame([y1]).T # Las respuestas

X.loc[x2.isna(), "Age"] = age_cluster.loc[clu.predict(Cp)]["Age"].values # Imputación de edades faltas por su edad mediana del clúster

X

,Sex,Age,Pclass2,Pclass2,Family,Family2,Family2,Fare2,Cabin2,Embarked2,Embarked2,Embarked2
0,0.0,22.0,0.0,0.0,1.0,1.0,0.0,2.110213,0.0,0.0,0.0,0.0
1,1.0,38.0,1.0,0.0,1.0,1.0,0.0,4.280593,1.0,1.0,0.0,0.0
2,1.0,26.0,0.0,0.0,0.0,0.0,0.0,2.188856,0.0,0.0,0.0,0.0
3,1.0,35.0,1.0,0.0,1.0,1.0,0.0,3.990834,1.0,0.0,0.0,0.0
4,0.0,35.0,0.0,0.0,0.0,0.0,0.0,2.202765,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,27.0,0.0,1.0,0.0,0.0,0.0,2.639057,0.0,0.0,0.0,0.0
887,1.0,19.0,1.0,0.0,0.0,0.0,0.0,3.433987,1.0,0.0,0.0,0.0
888,1.0,23.5,0.0,0.0,3.0,0.0,1.0,3.196630,0.0,0.0,0.0,0.0
889,0.0,26.0,1.0,0.0,0.0,0.0,0.0,3.433987,1.0,1.0,0.0,0.0
